# Day 6 · Vintage — แผนที่ที่ดูดาวแห่งพระราชอาณาจักรสยาม (สไตล์แผ่น engraving)

**#30DayMapChallenge 2026 · วันที่ 6 พ.ย. 2569** · ธีม *"do a map which looks like a historical map"*

ต่อยอด Day 1–5 โดยไม่คิดเลขใหม่ — เปลี่ยนแค่ **notation** ให้เป็นของยุคก่อน 1900:

| ของเดิม (สมัยใหม่) | ของวันนี้ (ย้อนยุค) |
|---|---|
| hillshade จาก DEM | เส้นขน **hachure** เดินตาม fall-line |
| choropleth หกเหลี่ยม H3 | **เส้นชั้น isopleth** วาดแบบสั่นมือ |
| polar plot เส้นขอบฟ้า | **rose 24 ดอก** ขีดทแยงแบบ engraving |
| กิโลเมตร | **เส้น / โยชน์** (40 ม. / 16 กม.) |
| พื้นหลังเรียบ | **กระดาษแก่ procedural** (คราบรา + รอยพับ + ขอบฉีก) |

> รัน cell ล่างสุดตามลำดับได้เลย สคริปต์นี้ตรวจว่าอยู่บนเครื่องตัวเองหรือบน Colab อัตโนมัติ
> (บน Colab จะดึง CSV/geojson/ฟอนต์จาก GitHub ตรง ๆ) — เวลาทั้งหมด ~5 นาที ที่เหลือคือโหลด DEM tile


In [ ]:
# -*- coding: utf-8 -*-
"""
Day 6 · Vintage — "แผนที่แสดงที่ดูดาวแห่งพระราชอาณาจักรสยาม"
#30DayMapChallenge 2026 · one map per day

ต่อจาก Day 1–5 โดยไม่คิดเลขใหม่ (ยกเว้นเส้นขอบฟ้าที่ re-run เพื่อยืนยันค่าเดิม):
  · 83 จุดดูดาว NARIT          -> หมึกจุดเล็กบนแผ่น
  · field ระยะทางสู่จุดใกล้สุด  -> เส้นชั้น isopleth "สั่นมือ" แทน choropleth หกเหลี่ยม
  · DEM terrarium z=9          -> เส้นขน hachure แทน hillshade
  · เส้นขอบฟ้า 360 ทิศ 24 จุด   -> แผ่น rose แยก (Planche)
กระดาษ/คราบ/ขอบฉีก สร้างด้วย noise ล้วน ไม่มี texture ไฟล์จากที่อื่น
"""
import io, os, json, math, time, zipfile, urllib.request
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.path import Path as MPath
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib import patheffects as pe
import geopandas as gpd
from shapely.geometry import box, MultiPolygon, Polygon
from shapely.ops import transform as shp_transform
from pyproj import CRS, Transformer

np.random.seed(2569)
# รันได้ทั้งบนเครื่องตัวเองและบน Colab: ถ้าไม่มีโฟลเดอร์รีโปให้ ก็ดึงไฟล์ข้อมูลจาก GitHub ตรง ๆ
COLAB = not os.path.isdir(r"D:\GIS30Days\geo-notes")
WORK  = os.getcwd() if COLAB else r"D:\GIS30Days"
REPO  = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day06") if not COLAB else os.path.join(WORK, "day06_cache")
OUT   = os.path.join(WORK, "out", "day06")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
RAW = "https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/"
if COLAB:
    for rel in ["data/darksky_thailand.csv", "posts/day05-sight/day05_sight.geojson",
                "ofl/trirong/Trirong-Regular.ttf", "ofl/trirong/Trirong-SemiBold.ttf",
                "ofl/trirong/Trirong-Bold.ttf", "ofl/trirong/Trirong-Italic.ttf",
                "ofl/trirong/Trirong-SemiBoldItalic.ttf", "ofl/chonburi/Chonburi-Regular.ttf"]:
        dst = os.path.join(CACHE, os.path.basename(rel))
        if not os.path.exists(dst):
            src = RAW + rel if rel.startswith(("data", "posts")) else "https://github.com/google/fonts/raw/main/" + rel
            urllib.request.urlretrieve(src, dst)
    print("Colab: ดึงข้อมูล + ฟอนต์ครบแล้ว")

PAPER = "#f3e7cd"; PAPER_D = "#e7d5ae"; INK = "#2b2114"; INK_SOFT = "#4b3a24"
RUST = "#8f4a24"; SEA = "#b9cbc9"; SEA_INK = "#4d6f74"; RIV = "#5d7f8b"
NEIGH = "#e0cda2"; DESK = "#241f1a"

def rgba(c, a=1.0):
    r, g, b = matplotlib.colors.to_rgb(c)
    return (r, g, b, a)

for f in sorted(os.listdir(CACHE)):
    if f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, f))
plt.rcParams.update({"font.family": "Trirong", "svg.fonttype": "none"})

TH_NUM = str.maketrans("0123456789", "".join(chr(0x0E50 + i) for i in range(10)))
def T(x):
    return str(x).translate(TH_NUM)

# ---------------------------------------------------------------- lint: Thai strings
COMBINING = {0x0E31, 0x0E33, 0x0E34, 0x0E35, 0x0E36, 0x0E37, 0x0E38, 0x0E39, 0x0E3A,
             0x0E47, 0x0E48, 0x0E49, 0x0E4A, 0x0E4B, 0x0E4C, 0x0E4D, 0x0E4E}
def lint(*strings):
    bad = []
    flat = []
    for s in strings:
        if isinstance(s, (list, tuple)):
            flat.extend(sum([list(x) if isinstance(x, (list, tuple)) else [x] for x in s], []))
        else:
            flat.append(s)
    for s in flat:
        for i, ch in enumerate(s):
            nxt = s[i + 1] if i + 1 < len(s) else ""
            # flag only "space + combining mark" = a mark stranded after a space (word cut in two)
            if ch == " " and nxt and ord(nxt) in COMBINING:
                bad.append(repr(s[max(0, i - 6):i + 6]))
    if bad:
        print("!! Thai string lint flagged:", bad)
    return bad



## 1) ข้อมูลจากรีโป — 83 จุด NARIT + 24 observer จาก Day 5

In [ ]:
# ---------------------------------------------------------------- 1. data
print("[1] อ่านข้อมูลรีโป ...")
df = pd.read_csv(os.path.join(REPO, "data", "darksky_thailand.csv"))
g5 = json.load(open(os.path.join(REPO, "posts/day05-sight/day05_sight.geojson"), encoding="utf-8"))
sites = [(float(r.latitude), float(r.longitude)) for r in df.itertuples()]
obs = [dict(rank=int(f["properties"]["rank"]), name=f["properties"]["name"],
            lon=f["geometry"]["coordinates"][0], lat=f["geometry"]["coordinates"][1],
            open_sky=float(f["properties"]["open_sky"]), mean_horizon=float(f["properties"]["mean_horizon"]),
            max_horizon=float(f["properties"]["max_horizon"])) for f in g5["features"]]
print(f"    NARIT {len(sites)} จุด · observer {len(obs)} จุด")

## 2) Basemap — Natural Earth 10m (ประเทศเพื่อนบ้าน + แม่น้ำ) *ต้อง clip ก่อน reproject*

In [ ]:
# ---------------------------------------------------------------- 2. basemaps
print("[2] Natural Earth 10m ...")
def ne_layer(stem, sub):
    shp = os.path.join(CACHE, stem + ".shp")
    if not os.path.exists(shp):
        zp = os.path.join(CACHE, stem + ".zip")
        urllib.request.urlretrieve(f"https://naciscdn.org/naturalearth/10m/{sub}/{stem}.zip", zp)
        with zipfile.ZipFile(zp) as z: z.extractall(CACHE)
    return gpd.read_file(shp)

countries = ne_layer("ne_10m_admin_0_countries", "cultural")
rivers    = ne_layer("ne_10m_rivers_lake_centerlines_scale_rank", "physical")
AOI = box(96, 4.0, 108.0, 22.0)
sel   = countries[countries.geometry.intersects(AOI)]
th    = sel[sel["ISO_A3"] == "THA"].geometry.iloc[0]
# clip ก่อน reproject: eqdc บิดมากเมื่ออยู่ไกล standard parallels ถ้าไม่ clip จะได้ inf/NaN
CTXBOX = box(96.5, 4.3, 107.6, 21.7)
neigh = sel[sel["ISO_A3"] != "THA"].copy()
neigh["geometry"] = neigh.geometry.intersection(AOI)
riv = rivers[rivers.geometry.intersects(AOI) & (rivers["scalerank"] <= 7)].copy()
riv["geometry"] = riv.geometry.intersection(CTXBOX)
riv = riv[~riv.geometry.is_empty]
print(f"    เพื่อนบ้าน {len(neigh)} · แม่น้ำ {len(riv)} เส้น")

## 3) Projection — Equidistant Conic ที่เส้นขนานโค้ง (ลุคแผ่นสำรวจยุคแรก)

In [ ]:
# ---------------------------------------------------------------- 3. projection
CRS_P = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 "
                       "+x_0=0 +y_0=0 +units=m +no_defs +ellps=WGS84")
TRF = Transformer.from_crs(CRS.from_epsg(4326), CRS_P, always_xy=True)
def proj(lonlat):
    a = np.asarray(lonlat, float)
    x, y = TRF.transform(a[:, 0], a[:, 1])
    return np.column_stack([x, y])
def proj_geom(g):
    return shp_transform(lambda x, y, z=None: TRF.transform(x, y), g)
def rings_of(geom):
    polys = geom.geoms if geom.geom_type.startswith("Multi") else [geom]
    for p in polys:
        for ring in [p.exterior] + list(p.interiors):
            a = np.asarray(ring.coords, float)
            a = a[np.isfinite(a).all(axis=1)]
            if len(a) > 3:
                yield a

thp = proj_geom(th)
X0, Y0, X1, Y1 = thp.bounds
PAD = 140_000
cx0, cx1, cy0, cy1 = X0 - PAD, X1 + PAD, Y0 - PAD, Y1 + PAD

## 4) DEM terrarium z=9 ทั้งประเทศ + land mask

In [ ]:
# ---------------------------------------------------------------- 4. DEM
print("[4] DEM terrarium ...")
Z = 9; NT = 2 ** Z; TPX = NT * 256
LON0, LON1, LAT0, LAT1 = 95.5, 108.5, 4.0, 22.0
XT0, XT1 = int((LON0 + 180) / 360 * NT), int((LON1 + 180) / 360 * NT)
YT0, YT1 = int((1 - math.asinh(math.tan(math.radians(LAT1))) / math.pi) / 2 * NT), \
           int((1 - math.asinh(math.tan(math.radians(LAT0))) / math.pi) / 2 * NT)
COLS, ROWS = XT1 - XT0 + 1, YT1 - YT0 + 1

def tile(z, x, y, tries=3):
    url = f"https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png"
    for k in range(tries):
        try:
            rgb = np.asarray(Image.open(io.BytesIO(urllib.request.urlopen(url, timeout=40).read())).convert("RGB"), np.float32)
            return rgb[:, :, 0] * 256 + rgb[:, :, 1] + rgb[:, :, 2] / 256.0 - 32768.0
        except Exception:
            if k == tries - 1:
                return np.full((256, 256), np.nan, np.float32)

npz = os.path.join(CACHE, f"dem_z{Z}.npz")
if os.path.exists(npz):
    DEM = np.load(npz)["H"]
else:
    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time(); DEM = np.full((ROWS * 256, COLS * 256), np.nan, np.float32)
    jobs = [(x, y) for x in range(XT0, XT1 + 1) for y in range(YT0, YT1 + 1)]
    def job(p):
        x, y = p; return (x - XT0, y - YT0, tile(Z, x, y))
    with ThreadPoolExecutor(12) as ex:
        for i, j, h in ex.map(job, jobs):
            DEM[j * 256:(j + 1) * 256, i * 256:(i + 1) * 256] = h
    np.savez_compressed(npz, H=DEM)
    print(f"    {len(jobs)} tiles ใน {time.time()-t0:.0f} วิ")
H_, W_ = DEM.shape
def col2lon(c): return (np.asarray(c) + XT0 * 256) / TPX * 360 - 180
def lon2col(lon): return (np.asarray(lon) + 180) / 360 * TPX - XT0 * 256
def row2lat(r):
    f = (np.asarray(r) + YT0 * 256) / TPX
    return np.degrees(np.arcsinh(np.tan(np.pi * (1 - 2 * f))))
def lat2row(lat):
    return (1 - np.arcsinh(np.tan(np.radians(np.asarray(lat)))) / math.pi) / 2 * TPX - YT0 * 256
RESX = 156543.0339 / NT
row_res = RESX * np.cos(np.radians(row2lat(np.arange(H_))))          # metres per pixel, per row
print(f"    grid {DEM.shape} · สูง {np.nanmin(DEM):.0f}–{np.nanmax(DEM):.0f} ม. · {row_res[H_//2]:.0f} ม./px")

# land mask by rasterising the polygon (fast, low memory)
im = Image.new("1", (W_, H_), 0); dr = ImageDraw.Draw(im)
for poly in (th.geoms if th.geom_type.startswith("Multi") else [th]):
    ext = list(zip(lon2col(np.asarray(poly.exterior.coords.xy[0])), lat2row(np.asarray(poly.exterior.coords.xy[1]))))
    dr.polygon(ext, fill=1)
    for h in poly.interiors:
        hole = list(zip(lon2col(np.asarray(h.coords.xy[0])), lat2row(np.asarray(h.coords.xy[1]))))
        dr.polygon(hole, fill=0)
LAND = np.array(im, dtype=bool)
print(f"    land mask = {LAND.mean():.3f} ของกริด")

## 5) wobble — noise เรียบที่ทำให้เส้นดูเขียนด้วยมือ

In [ ]:
# ---------------------------------------------------------------- 5. wobble
def smooth_noise(n, amp_m, seed=0):
    rng = np.random.default_rng(seed); s = np.zeros(n)
    for lam, w in ((90., .55), (35., .3), (13., .15)):
        s += w * np.sin(2 * np.pi * np.arange(n) / lam + rng.uniform(0, 2 * np.pi))
    s -= s.mean(); s /= (np.abs(s).max() + 1e-9)
    return s * amp_m

def wobble(P, amp_m=2200.0, res_m=6000.0, seed=0, close=False):
    P = np.asarray(P, float)
    if len(P) < 3: return P
    if close: P = P[:-1]
    seg = np.hypot(*np.diff(P, axis=0).T)
    cum = np.concatenate([[0], np.cumsum(seg)])
    n = max(int(cum[-1] / res_m), 6)
    ds = np.linspace(0, cum[-1], n)
    Q = np.column_stack([np.interp(ds, cum, P[:, 0]), np.interp(ds, cum, P[:, 1])])
    d = np.gradient(Q, axis=0); L = np.hypot(d[:, 0], d[:, 1]) + 1e-9
    Q = Q + np.column_stack([-d[:, 1] / L, d[:, 0] / L]) * smooth_noise(n, amp_m, seed)[:, None]
    return np.vstack([Q, Q[:1]]) if close else Q

## 6) hachure — เกลี่ย DEM ก่อน แล้วเดิน fall-line หยุดเมื่อทิศกลับ

In [ ]:
# ---------------------------------------------------------------- 6. hachures
print("[6] เส้นขน hachure จาก DEM ...")
def hachures(seed_px=15, step_px=2.6, min_slope=4.0, max_len=18, lw_a=0.22, lw_b=0.030, lw_cap=0.90):
    fill = np.nan_to_num(DEM, nan=0.0)
    # เกลี่ย DEM ก่อน (sigma 2 px ~ 600 ม.) ไม่งั้น gradient ต่อ pixel ส่ายไปมา -> เส้นกลายเป็นก้นหอย
    smooth = gaussian_filter(fill, 2.0)
    g_row, g_col = np.gradient(smooth)
    gx = g_col / row_res[:, None]; gy = g_row / row_res[:, None]
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    ok = LAND & (slope >= min_slope)
    cs, rs = np.meshgrid(np.arange(0, W_, seed_px), np.arange(0, H_, seed_px))
    cs, rs = cs.ravel(), rs.ravel()
    m = ok[rs, cs]; cs, rs = cs[m].astype(float), rs[m].astype(float)
    strokes = []
    for c0, r0 in zip(cs, rs):
        c, r = c0, r0; pts = [(c, r)]; pc, pr = c, r; last = None
        for _ in range(max_len):
            i, j = int(round(r)), int(round(c))
            if not (1 <= i < H_ - 1 and 1 <= j < W_ - 1): break
            L = math.hypot(gx[i, j], gy[i, j])
            if L < 1e-9: break
            dc, dr = -gx[i, j] / L, -gy[i, j] / L
            if last is not None and dc * last[0] + dr * last[1] < 0.2: break   # ทิศกลับ = ถึงตีนเขา
            c += dc * step_px; r += dr * step_px
            if not LAND[int(round(r)), int(round(c))]: break
            pts.append((c, r)); last = (dc, dr)
        if len(pts) < 7 or math.hypot(c - pc, r - pr) < 0.55 * len(pts) * step_px: continue
        w = min(lw_a + lw_b * slope[int(round(r0)), int(round(c0))], lw_cap)
        P = proj(np.column_stack([col2lon([p[0] for p in pts]), row2lat([p[1] for p in pts])]))
        k = len(P) // 2
        strokes.append((P[:k + 1], w))
        if len(P) - k > 2: strokes.append((P[k:], w * 0.5))
    print(f"    {len(strokes)} เส้นขน จาก seed ที่ชันพอ {len(cs)} จุด")
    return strokes
HACH = hachures()

## 7) isopleth — KD-tree ระยะทางสู่จุดฟ้ามืด → เส้นชั้น 7 ระดับ

In [ ]:
# ---------------------------------------------------------------- 7. isopleths
print("[7] เส้นชั้นระยะทางสู่จุดฟ้ามืด ...")
LAT0K = 13.0
def km(lon, lat):
    return np.column_stack([np.asarray(lon) * 111.320 * math.cos(math.radians(LAT0K)),
                            np.asarray(lat) * 110.574])
tree = cKDTree(km([s[1] for s in sites], [s[0] for s in sites]))
lons = np.arange(97.0, 106.3, 0.02); lats = np.arange(5.3, 20.7, 0.02)
GC, GR = np.meshgrid(lons, lats)
dist = tree.query(km(GC.ravel(), GR.ravel()), k=1)[0].reshape(GC.shape)
dist[~LAND[np.clip(lat2row(GR).astype(int), 0, H_ - 1), np.clip(lon2col(GC).astype(int), 0, W_ - 1)]] = np.nan
LEV = [10, 20, 30, 50, 75, 100, 150]
tmp = plt.figure(); tax = tmp.add_subplot(111); CS = tax.contour(GC, GR, dist, levels=LEV); plt.close(tmp)
iso = []
for li, level in enumerate(CS.levels):
    for si, seg in enumerate(CS.allsegs[li]):
        if len(seg) < 8: continue
        iso.append((wobble(proj(seg), 3000 if li % 3 == 2 else 1700, 5000, seed=li * 97 + si * 13), level))
print(f"    {len(iso)} ช่วงเส้น · ระดับ {LEV} กม.")

## 8) กระดาษแก่ procedural (mottle + foxing + รอยพับ + vignette + ขอบฉีก)

In [ ]:
# ---------------------------------------------------------------- 8. paper
print("[8] กระดาษแก่ procedural ...")
def paper_rgba(W, Hh, seed=7):
    rng = np.random.default_rng(seed)
    base = np.array(matplotlib.colors.to_rgb(PAPER))
    dark = np.array(matplotlib.colors.to_rgb("#d6bf8e"))
    spot = np.array(matplotlib.colors.to_rgb("#a8794a"))
    y, x = np.mgrid[0:Hh, 0:W].astype(float)
    mottle = gaussian_filter(rng.standard_normal((Hh, W)), Hh / 20)
    mottle = (mottle - mottle.min()) / (np.ptp(mottle) + 1e-9)
    fine = gaussian_filter(rng.standard_normal((Hh, W)), 1.1)
    img = base[None, None, :] * (1 - 0.16 * mottle[..., None]) + 0.22 * mottle[..., None] * (dark - base)
    img += 0.018 * (fine[..., None] - 0.0) * base
    for _ in range(36):                                       # foxing / ราจุดน้ำตาล
        cy, cx = rng.integers(0, Hh), rng.integers(0, W)
        rr = rng.uniform(Hh * 0.004, Hh * 0.020)
        d2 = (y - cy) ** 2 + (x - cx) ** 2
        ring = np.exp(-d2 / (2 * rr ** 2)) * (1 - 0.5 * np.exp(-d2 / (2 * (rr * 0.4) ** 2)))
        img = img * (1 - 0.20 * ring[..., None]) + 0.20 * ring[..., None] * spot
    for frac, axis in ((0.37, "v"), (0.56, "h")):             # รอยพับ
        p = (x / W) if axis == "v" else (y / Hh)
        band = np.exp(-((p - frac) ** 2) / (2 * 0.0022 ** 2))
        img = img * (1 - 0.055 * band[..., None]) + 0.055 * band[..., None] * np.array([0.60, 0.53, 0.42])
    R = np.hypot(x / W - .5, (y / Hh - .5) * (Hh / W) * 0.62)
    img *= (1 - 0.17 * np.clip((R - 0.30) / 0.52, 0, 1) ** 1.3)[..., None]
    ang = np.arctan2(y - Hh / 2, x - W / 2)
    wob = sum(a * np.sin(k * ang + rng.uniform(0, 6.283)) for k, a in ((5, .006), (11, .0035), (23, .0022), (41, .0012)))
    rx = W * .5 * .975 + wob * W; ry = Hh * .5 * .975 + wob * Hh
    p_ = 9.0                                    # superellipse -> sheet with rounded torn corners
    inside = (((np.abs(x - W / 2) / rx) ** p_) + (np.abs(y - Hh / 2) / ry) ** p_) <= 1.0
    alpha = np.clip(gaussian_filter(inside.astype(float), 1.8) * 1.04, 0, 1)
    return (np.dstack([np.clip(img, 0, 1), alpha[..., None]]) * 255).astype(np.uint8)

## 9) เส้นขอบฟ้า 24 จุด (วิธีเดียวกับ Day 5) + ยืนยันค่าเดิม

In [ ]:
# ---------------------------------------------------------------- 9. horizon roses
print("[9] คำนวณเส้นขอบฟ้า 24 จุด (วิธีเดียวกับ Day 5) ...")
Z13 = 13
def win13(lat, lon, span=1):
    n = 2 ** Z13
    gxf = (lon + 180) / 360 * n; gyf = (1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n
    xt0, yt0 = int(gxf), int(gyf); size = 2 * span + 1
    BIG = np.full((size * 256, size * 256), np.nan, float)
    for dx in range(-span, span + 1):
        for dy in range(-span, span + 1):
            BIG[(dy + span) * 256:(dy + span + 1) * 256, (dx + span) * 256:(dx + span + 1) * 256] = tile(Z13, xt0 + dx, yt0 + dy)
    return BIG, 156543.0339 * math.cos(math.radians(lat)) / n, \
        int(round((gxf - (xt0 - span)) * 256)), int(round((gyf - (yt0 - span)) * 256))

def sky_viewshed(dem, res_m, opx, opy, eye=2.0):
    Hh, Wd = dem.shape; hz = np.zeros(360)
    opy = min(max(opy, 0), Hh - 1); opx = min(max(opx, 0), Wd - 1)
    z = float(np.nanmean(dem[max(0, opy - 1):opy + 2, max(0, opx - 1):opx + 2])) + eye
    for a in range(360):
        th_ = math.radians(a); cx_, cy_ = math.sin(th_), math.cos(th_)
        hm = -90.0; r = 1.0; maxr = int(min(Hh, Wd) / 2) - 1
        while r < maxr:
            xx = int(round(opx + cx_ * r)); yy = int(round(opy - cy_ * r))
            if 0 <= xx < Wd and 0 <= yy < Hh:
                v = dem[yy, xx]
                if not np.isnan(v):
                    ang = math.degrees(math.atan2(v - z, r * res_m))
                    if ang > hm: hm = ang
            r += 1.0
        hz[a] = hm
    return hz

hor_npy = os.path.join(CACHE, "horizons24.npz")
if os.path.exists(hor_npy):
    HZ = np.load(hor_npy)["H"]
else:
    t0 = time.time(); HZ = np.zeros((len(obs), 360))
    for i, s in enumerate(obs):
        dem, res, ox, oy = win13(s["lat"], s["lon"])
        HZ[i] = sky_viewshed(dem, res, ox, oy)
    np.savez_compressed(hor_npy, H=HZ)
    print(f"    ใช้เวลา {time.time()-t0:.0f} วิ")
chk = np.clip(HZ, 0, 90).mean(axis=1); ref = np.array([s["mean_horizon"] for s in obs])
print(f"    ยืนยันกับ Day 5: mean|Δ| = {np.abs(chk-ref).mean():.3f}° · max|Δ| = {np.abs(chk-ref).max():.3f}°")

## 10) ประกอบแผ่นแผนที่ (cartouche, rose, เส้นชั้น, hachure, มาตราเส้น/โยชน์)

In [ ]:
# ---------------------------------------------------------------- 10. plate
print("[10] ประกอบแผ่นแผนที่ ...")
FIG_W, FIG_H, DPI = 14.0, 18.0, 170
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor=DESK)
ax_bg = fig.add_axes([0.010, 0.008, 0.98, 0.984]); ax_bg.axis("off"); ax_bg.set_zorder(0)
ax_bg.imshow(paper_rgba(int(FIG_W * DPI * 0.6), int(FIG_H * DPI * 0.6)), extent=[0, 1, 0, 1],
             origin="lower", interpolation="bilinear", aspect="auto")

BOX = dict(x=0.055, y=0.118, w=0.89, h=0.772)          # map zone in figure fraction
S = min(BOX["w"] * FIG_W / (cx1 - cx0), BOX["h"] * FIG_H / (cy1 - cy0))   # inches per metre
mw, mh = (cx1 - cx0) * S / FIG_W, (cy1 - cy0) * S / FIG_H                 # axes size in FRACTION
axm = fig.add_axes([BOX["x"] + (BOX["w"] - mw) / 2, BOX["y"] + (BOX["h"] - mh) / 2, mw, mh])
axm.set_zorder(2); axm.set_xlim(cx0, cx1); axm.set_ylim(cy0, cy1); axm.set_aspect("equal")
axm.set_facecolor("none"); axm.set_xticks([]); axm.set_yticks([])
for sp in axm.spines.values(): sp.set_visible(False)
MPP = (cx1 - cx0) / (mw * FIG_W * DPI)      # metres per figure pixel
AXM_RECT = axm.get_position().bounds          # (x0, y0, w, h) in figure fraction — captured before resize/close

# ทะเล: wash + เส้นคลื่นขนานฝั่ง
for k, (buf, a) in enumerate([(9_000, .50), (23_000, .34), (42_000, .22), (64_000, .14)]):
    segs = []
    for r in rings_of(thp):
        b = Polygon(r).buffer(buf)
        for rr in rings_of(b): segs.append(rr)
    axm.add_collection(PolyCollection(segs, facecolors="none", edgecolors=rgba(SEA_INK, a),
                                     linewidths=0.45, zorder=3.0 + k * 0.01))
land_rings = list(rings_of(thp))
axm.add_collection(PolyCollection(land_rings, facecolors=rgba(SEA, .5), edgecolors="none", zorder=2.5))
# ประเทศเพื่อนบ้าน (ย้อมมือ — ตั้งใจให้เลื้อยจากเส้นหมึกนิดหน่อย)
n_rings = [r for g in neigh.geometry.map(proj_geom) for r in rings_of(g)]
axm.add_collection(PolyCollection([r + np.array([4 * MPP, -4 * MPP]) for r in n_rings],
                                 facecolors=rgba(NEIGH, .8), edgecolors="none", zorder=2.6))
axm.add_collection(LineCollection([wobble(r, 900, 4500, seed=77, close=True) for r in n_rings],
                                  colors=rgba(INK, .45), linewidths=0.55, zorder=2.65))
axm.add_collection(PolyCollection(land_rings, facecolors=rgba(PAPER_D, .95), edgecolors="none", zorder=2.7))

# ลายกริดลองจิจูด/ละติจูด
grat = [proj(np.column_stack([np.full(90, lo), np.linspace(3.0, 23.0, 90)])) for lo in range(96, 108, 2)]
grat += [proj(np.column_stack([np.linspace(94, 110, 90), np.full(90, la)])) for la in range(4, 23, 2)]
axm.add_collection(LineCollection(grat, colors=rgba(INK, .17), linewidths=0.45, zorder=3.1))

# เส้นขนภูมิประเทศ
axm.add_collection(LineCollection([p for p, w in HACH], colors=rgba(INK, .58),
                                  linewidths=[w for p, w in HACH], capstyle="round", zorder=3.4))
# แม่น้ำ
rv = []
for i, g in enumerate(riv.geometry.map(proj_geom)):
    for rr in (g.geoms if g.geom_type.startswith("Multi") else [g]):
        c = np.asarray(rr.coords, float)
        if len(c) > 4: rv.append(wobble(c, 1100, 5000, seed=101 + i))
axm.add_collection(LineCollection(rv, colors=rgba(INK, .30), linewidths=1.7, zorder=3.5))
axm.add_collection(LineCollection(rv, colors=rgba(RIV, .85), linewidths=0.7, zorder=3.6))
# เส้นชั้นระยะทาง
idx = [i for i, (p, lv) in enumerate(iso) if lv in (50, 150)]
oth = [i for i in range(len(iso)) if i not in set(idx)]
axm.add_collection(LineCollection([iso[i][0] for i in oth], colors=rgba(INK_SOFT, .55), linewidths=0.5, zorder=3.8))
axm.add_collection(LineCollection([iso[i][0] for i in idx], colors=rgba(RUST, .85), linewidths=1.1, zorder=3.9))
halo = [pe.withStroke(linewidth=2.4, foreground=rgba(PAPER, .95))]
for lv in (50, 150):
    best = max([p for p, l in iso if l == lv], key=lambda p: np.hypot(*np.diff(p, axis=0).T).sum())
    m = len(best) // 2
    ang = math.degrees(math.atan2(best[m + 1][1] - best[m][1], best[m + 1][0] - best[m][0]))
    axm.text(best[m][0], best[m][1], T(lv), fontsize=7.4, color=RUST, rotation=ang,
             rotation_mode="anchor", ha="center", va="center", path_effects=halo, zorder=4.0)
# เส้นฝั่ง (หมึก双线)
coast = [wobble(r, 1100, 4500, seed=3, close=True) for r in rings_of(thp)]
axm.add_collection(LineCollection(coast, colors=rgba(INK, .95), linewidths=1.2, zorder=4.2))
axm.add_collection(LineCollection([c + np.array([1.8 * MPP, -1.8 * MPP]) for c in coast],
                                  colors=rgba(INK, .28), linewidths=0.5, zorder=4.1))
# จุดข้อมูล
sp_ = proj(np.array(sites)); op = proj([[s["lon"], s["lat"]] for s in obs])
axm.scatter(sp_[:, 0], sp_[:, 1], s=6, marker=".", color=rgba(INK, .7), zorder=5.0)
axm.scatter(op[:, 0], op[:, 1], s=120, facecolors=rgba(PAPER, .95), edgecolors=RUST, linewidths=1.1, zorder=5.1)
for (x, y), s in zip(op, obs):
    axm.text(x, y, T(s["rank"]), fontsize=6.3, color=INK, ha="center", va="center", zorder=5.2)
for (x, y), s in zip(op, sorted(obs, key=lambda z: -z["open_sky"])[:6]):
    axm.annotate(s["name"][:26], (x, y), textcoords="offset points", xytext=(9, 7),
                 fontsize=6.5, color=INK_SOFT, path_effects=halo, zorder=5.3)

# เข็มทิศในอ่าวไทย
def rose(ax, cx_, cy_, R):
    for i in range(32):
        a = math.radians(i * 11.25)
        r1, r2 = ((R, R * .40) if i % 4 == 0 else (R * .70, R * .28) if i % 2 == 0 else (R * .48, R * .20))
        pts = [(cx_ + r1 * math.cos(a), cy_ + r1 * math.sin(a)),
               (cx_ + r2 * math.cos(a - math.pi / 2), cy_ + r2 * math.sin(a - math.pi / 2)),
               (cx_, cy_),
               (cx_ + r2 * math.cos(a + math.pi / 2), cy_ + r2 * math.sin(a + math.pi / 2))]
        ax.add_collection(PolyCollection([pts], facecolors=[rgba(INK, .72) if i % 4 == 0 else rgba(PAPER, .55)],
                                        edgecolors=rgba(INK, .85), linewidths=0.35, zorder=5.6))
    for rr, a in ((R * 1.05, .8), (R * .30, .6)):
        ax.add_patch(plt.Circle((cx_, cy_), rr, fill=False, ec=rgba(INK, a), lw=0.6, zorder=5.6))
    for lab, a in (("อุดร", 90), ("บูรพา", 0), ("ทักษิณ", 270), ("ประจิม", 180)):
        ax.text(cx_ + R * 1.22 * math.cos(math.radians(a)), cy_ + R * 1.22 * math.sin(math.radians(a)),
                lab, fontsize=7.0, color=INK, ha="center", va="center", zorder=5.7)
rose(axm, *proj([[103.0, 8.4]])[0], R=92_000)

# ขอบแผ่น
for inset, lw in ((25_000, 1.4), (45_000, 0.5)):
    axm.add_patch(Rectangle((cx0 + inset, cy0 + inset), (cx1 - cx0) - 2 * inset, (cy1 - cy0) - 2 * inset,
                            fill=False, ec=rgba(INK, .9), lw=lw, zorder=6.0))
for a_ in ((cx0 + 25_000, cy0 + 25_000), (cx1 - 25_000, cy0 + 25_000), (cx0 + 25_000, cy1 - 25_000), (cx1 - 25_000, cy1 - 25_000)):
    d = 22_000
    axm.add_patch(plt.Polygon([(a_[0], a_[1] + d), (a_[0] - d, a_[1]), (a_[0], a_[1] - d), (a_[0] + d, a_[1])],
                              closed=True, facecolor=rgba(INK, .85), edgecolor="none", zorder=6.1))
# ตัดทุก layer ของแผนที่ให้อยู่ในกรอบ ( margin ไว้ใส่เลขลองจิจูด )
frame_in = 25_000
clip_patch = Rectangle((cx0 + frame_in, cy0 + frame_in), (cx1 - cx0) - 2 * frame_in, (cy1 - cy0) - 2 * frame_in,
                       transform=axm.transData, fc="none", ec="none", zorder=1)
axm.add_patch(clip_patch)
for coll in list(axm.collections) + list(axm.patches):
    if coll is not clip_patch and coll.get_zorder() < 6.0:
        coll.set_clip_path(clip_patch)
for lo in range(98, 107, 2):
    x = proj([[lo, 3.0]])[0][0]
    if cx0 < x < cx1: axm.text(x, cy1 - 12_000, f"{lo}°", fontsize=7, color=INK, ha="center", va="bottom", zorder=6.2)
for la in range(6, 21, 2):
    y = proj([[94, la]])[0][1]
    if cy0 < y < cy1: axm.text(cx0 + 14_000, y, f"{la}°", fontsize=7, color=INK, ha="left", va="center", zorder=6.2)

# cartouche
TXT = dict(
    title="แผนที่แสดงที่ดูดาวแห่งพระราชอาณาจักรสยาม",
    sub="เส้นชั้นแสดงระยะทางสู่จุดอนุรักษ์ท้องฟ้ามืดที่ใกล้ที่สุด พร้อมเส้นขนแสดงภูมิประเทศ",
    day="วันที่ ๖ · ธีม Vintage · #30DayMapChallenge 2026",
    leg_head="คำอธิบาย",
    leg=[("·", "จุดอนุรักษ์ท้องฟ้ามืดของ NARIT ๘๓ แห่ง"),
         ("๐", "hotspot ๒๔ กลุ่ม · ตัวเลข = อันดับจาก Day 4"),
         ("—", "เส้นชั้น ๕๐ และ ๑๕๐ กิโลเมตร"),
         ("////", "เส้นขน (hachure) แสดงพื้นที่ลาดชัน"),
         ("~", "แม่น้ำ จาก Natural Earth")],
    scale_head="มาตราส่วน",
    scale_note="๑ เส้น = ๔๐ เมตร · ๑ โยชน์ = ๑๖ กิโลเมตร",
    cred_head="แหล่งข้อมูล",
    cred=("จุดฟ้ามืด: สถาบันวิจัยดาราศาสตร์แห่งชาติ (NARIT)\n"
          "ภูมิประเทศ: Copernicus GLO-30 ผ่าน terrarium tiles ของ AWS\n"
          "ขอบประเทศและแม่น้ำ: Natural Earth (public domain)\n"
          "ตัวอักษร: Trirong และ Chonburi (SIL Open Font License)"),
    note="แบบย้อนยุคในแผ่นนี้เป็นงานออกแบบใหม่ ไม่ใช่แผนที่โบราณของจริง",
)
lint(TXT["title"], TXT["sub"], TXT["day"], TXT["leg_head"], TXT["scale_head"],
                TXT["scale_note"], TXT["cred_head"], TXT["cred"], TXT["note"], *TXT["leg"])

axc = fig.add_axes([0.26, 0.902, 0.48, 0.076]); axc.axis("off"); axc.set_zorder(8)
axc.set_xlim(0, 1); axc.set_ylim(0, 1)
axc.add_patch(FancyBboxPatch((.004, .04), .992, .92, boxstyle="round,pad=0.004,rounding_size=.012",
                             fc=rgba(PAPER, .30), ec=rgba(INK, .85), lw=1.3))
axc.add_patch(FancyBboxPatch((.018, .12), .964, .76, boxstyle="round,pad=0.004,rounding_size=.008",
                             fc="none", ec=rgba(INK, .5), lw=0.6))
axc.text(.5, .70, TXT["title"], fontsize=19, family="Chonburi", color=INK, ha="center", va="center")
axc.text(.5, .41, TXT["sub"], fontsize=9.2, color=INK_SOFT, ha="center", va="center")
axc.text(.5, .18, TXT["day"], fontsize=8.0, color=RUST, ha="center", va="center")
axc.plot([.32, .68], [.545, .545], color=rgba(INK, .45), lw=0.6)

ay = 0.012
axl = fig.add_axes([0.055, ay, 0.30, 0.092]); axl.axis("off"); axl.set_zorder(8); axl.set_xlim(0, 1); axl.set_ylim(0, 1)
axl.add_patch(Rectangle((0, 0), 1, 1, fc=rgba(PAPER, .28), ec=rgba(INK, .7), lw=0.9))
axl.text(.03, .89, TXT["leg_head"], fontsize=9, family="Chonburi", color=INK, va="center")
for i, (sym, txt) in enumerate(TXT["leg"]):
    yy = .71 - i * .16
    axl.text(.05, yy, sym, fontsize=9.5, color=RUST if i in (1, 2) else INK, va="center")
    axl.text(.13, yy, txt, fontsize=7.4, color=INK_SOFT, va="center")

axs = fig.add_axes([0.375, ay, 0.24, 0.092]); axs.axis("off"); axs.set_zorder(8); axs.set_xlim(0, 1); axs.set_ylim(0, 1)
axs.add_patch(Rectangle((0, 0), 1, 1, fc=rgba(PAPER, .28), ec=rgba(INK, .7), lw=0.9))
axs.text(.5, .86, TXT["scale_head"], fontsize=9, family="Chonburi", color=INK, ha="center", va="center")
bx, bw = .10, .80
for i in range(4):
    axs.add_patch(Rectangle((bx + i * bw / 4, .44), bw / 4, .15, fc=rgba(INK, .85) if i % 2 == 0 else rgba(PAPER, .55),
                            ec=rgba(INK, .85), lw=0.6))
for i, kmv in enumerate((0, 40, 80, 120, 160)):
    axs.text(bx + i * bw / 4, .65, T(kmv), fontsize=7.3, color=INK, ha="center", va="bottom")
axs.text(bx + bw / 4, .34, "๑,๐๐๐ เส้น", fontsize=6.4, color=INK_SOFT, ha="center", va="top")
axs.text(bx + bw, .34, "๑๐ โยชน์", fontsize=6.4, color=INK_SOFT, ha="right", va="top")
axs.text(.5, .13, TXT["scale_note"], fontsize=6.1, color=INK_SOFT, ha="center", va="center")

axr = fig.add_axes([0.635, ay, 0.31, 0.092]); axr.axis("off"); axr.set_zorder(8); axr.set_xlim(0, 1); axr.set_ylim(0, 1)
axr.add_patch(Rectangle((0, 0), 1, 1, fc=rgba(PAPER, .28), ec=rgba(INK, .7), lw=0.9))
axr.text(.03, .89, TXT["cred_head"], fontsize=9, family="Chonburi", color=INK, va="center")
axr.text(.03, .76, TXT["cred"], fontsize=5.9, color=INK_SOFT, va="top", linespacing=1.55)
axr.text(.03, .06, TXT["note"], fontsize=5.7, color=RUST, va="bottom")

for fx_, fy_, ha in ((0.024, 0.982, "left"), (0.976, 0.982, "right"), (0.024, 0.006, "left"), (0.976, 0.006, "right")):
    fig.text(fx_, fy_, "๏ ๖", fontsize=10, color=rgba(INK, .7), ha=ha, va="center", zorder=9)

def stain_overlay(W, Hh, seed=21):
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:Hh, 0:W].astype(float)
    a = np.zeros((Hh, W))
    for _ in range(26):
        cy, cx = rng.uniform(0, Hh), rng.uniform(0, W)
        rr = rng.uniform(Hh * 0.02, Hh * 0.10)
        a += rng.uniform(0.10, 0.30) * np.exp(-(((y - cy) ** 2 + (x - cx) ** 2) / (2 * rr ** 2)))
    a += 0.05 * gaussian_filter(rng.standard_normal((Hh, W)), Hh / 30)
    a = np.clip(a, 0, 0.55) * 0.42
    tint = np.array(matplotlib.colors.to_rgb("#8a6a3a"))
    img = np.zeros((Hh, W, 4))
    img[..., :3] = tint; img[..., 3] = a
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)

ax_st = fig.add_axes([0.010, 0.008, 0.98, 0.984]); ax_st.axis("off"); ax_st.set_zorder(7.2)
ax_st.imshow(stain_overlay(int(FIG_W * DPI * 0.5), int(FIG_H * DPI * 0.5)), extent=[0, 1, 0, 1],
             origin="lower", interpolation="bilinear", aspect="auto")

png = os.path.join(OUT, "day06_vintage_plate.png")
fig.savefig(png, dpi=DPI, facecolor=DESK)
print(f"    wrote {png} ({os.path.getsize(png)/1e6:.2f} MB)")
fig.set_size_inches(10.5, 13.5)
prev = os.path.join(OUT, "day06_vintage_preview.png")
fig.savefig(prev, dpi=112, facecolor=DESK)
print(f"    wrote {prev} ({os.path.getsize(prev)/1e6:.2f} MB)")
plt.close(fig)

## 11) crop รายละเอียดจากแผ่นความละเอียดสูง

In [ ]:
# ---------------------------------------------------------------- 11. detail crops
print("[11] crop รายละเอียด ...")
hi = os.path.join(CACHE, "plate_hi.png")
fig_hi = None
DPI_HI = 320
fig.set_size_inches(FIG_W, FIG_H)
fig.savefig(hi, dpi=DPI_HI, facecolor=DESK)
im_big = Image.open(hi); PW, PH = im_big.size
def fig_px(fx, fy):
    return int(fx * PW), int((1 - fy) * PH)
def crop_rect(fx0, fy0, fx1, fy1, name):
    left, right = fx0 * PW, fx1 * PW
    top, bottom = (1 - fy1) * PH, (1 - fy0) * PH
    c = im_big.crop((int(max(0, left)), int(max(0, top)), int(min(PW, right)), int(min(PH, bottom))))
    c.save(os.path.join(OUT, name)); print("    ", name, c.size)
def data_to_fig(lon, lat, pad_km=110):
    rx0, ry0, rw, rh = AXM_RECT
    x, y = proj([[lon, lat]])[0]
    fx = rx0 + (x - cx0) / (cx1 - cx0) * rw
    fy = ry0 + (y - cy0) / (cy1 - cy0) * rh
    d = pad_km * 1000 / (cx1 - cx0) * rw
    return fx - d, fy - d, fx + d, fy + d
crop_rect(0.26, 0.900, 0.74, 0.980, "day06_detail_cartouche.png")
crop_rect(*data_to_fig(98.6, 18.4, 120), name="day06_detail_hachure.png")
crop_rect(*data_to_fig(103.0, 8.4, 120), name="day06_detail_rose.png")
plt.close("all")

## 12) แผ่น rose เส้นขอบฟ้า 24 ดอก

In [ ]:
# ---------------------------------------------------------------- 12. roses plate
print("[12] แผ่น rose เส้นขอบฟ้า ...")
fig2 = plt.figure(figsize=(13, 8.4), facecolor=PAPER)
fig2.text(0.5, 0.968, "แผ่นที่ ๖ · เส้นขอบฟ้า ๓๖๐ ทิศของจุดดูดาว ๒๔ แห่ง",
          fontsize=15, family="Chonburi", color=INK, ha="center")
fig2.text(0.5, 0.933, "รัศมี = มุมที่ยอดเขาด้านข้างบังสายตา (จุดกลาง = 0° ขอบนอก = 25°) · วงเล็กแปลว่าฟ้าเปิดโล่งรอบทิศ",
          fontsize=8.4, color=INK_SOFT, ha="center")
for i, s in enumerate(obs):
    ax = fig2.add_subplot(4, 6, i + 1)
    ax.set_facecolor("none")
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    HSC = 25.0                                   # มุมขอบฟ้า 25° = ขอบนอกสุดของวง
    h = np.clip(HZ[i], 0, HSC); ang = np.radians(np.arange(361) % 360); hh = np.r_[h, h[0]]
    R = 0.30 + hh / HSC * 1.30
    P = np.column_stack([R * np.cos(ang), R * np.sin(ang)])
    ax.add_collection(PolyCollection([P], facecolors=rgba(PAPER_D, .8), edgecolors=rgba(INK, .95),
                                     linewidths=0.8, hatch="//"))
    for deg in (5, 15):
        ax.add_patch(plt.Circle((0, 0), 0.30 + deg / HSC * 1.30, fill=False, ec=rgba(INK, .35), lw=0.4, ls=(0, (2, 2))))
    for a in range(0, 360, 45):
        ax.plot([0.30 * math.cos(math.radians(a)), 1.62 * math.cos(math.radians(a))],
                [0.30 * math.sin(math.radians(a)), 1.62 * math.sin(math.radians(a))],
                color=rgba(INK, .22), lw=0.35)
    ax.text(0, 1.78, "น", fontsize=6.2, color=INK, ha="center", va="center")
    ax.set_xlim(-1.95, 1.95); ax.set_ylim(-2.0, 1.95); ax.set_aspect("equal")
    ax.set_title(f"{T(s['rank'])}  " + s["name"][:20], fontsize=7.2, color=INK, pad=3)
    ax.text(0, -1.92, f"เปิด {s['open_sky']:.3f} · สูงสุด {s['max_horizon']:.0f}°",
            fontsize=5.9, color=INK_SOFT, ha="center")
fig2.tight_layout(rect=(0, 0.015, 1, 0.915))
rp = os.path.join(OUT, "day06_roses_plate.png")
fig2.savefig(rp, dpi=150, facecolor=PAPER); plt.close(fig2)
print(f"    wrote {rp} ({os.path.getsize(rp)/1e6:.2f} MB)")

## 13) เส้นหมึกอย่างเดียว → SVG (เก็บไว้ขึ้น pen plotter วัน Pen & paper)

In [ ]:
# ---------------------------------------------------------------- 13. line-art SVG (plotter-friendly)
print("[13] เส้นหมึกอย่างเดียวสำหรับ SVG/plotter ...")
HACH2 = hachures(seed_px=26)
fig3 = plt.figure(figsize=(10.5, 13.5), facecolor="white")
ax3 = fig3.add_axes([0.03, 0.03, 0.94, 0.94])
ax3.set_xlim(cx0, cx1); ax3.set_ylim(cy0, cy1); ax3.set_aspect("equal"); ax3.axis("off")
ax3.add_collection(LineCollection([p for p, w in HACH2], colors=INK, linewidths=0.3))
ax3.add_collection(LineCollection([p for p, l in iso], colors=INK, linewidths=0.4))
ax3.add_collection(LineCollection(rv, colors=INK, linewidths=0.4))
ax3.add_collection(LineCollection(coast, colors=INK, linewidths=0.8))
ax3.scatter(op[:, 0], op[:, 1], s=60, facecolors="none", edgecolors=INK, linewidths=0.6)
svg = os.path.join(OUT, "day06_vintage_lineart.svg")
fig3.savefig(svg, facecolor="white")
print(f"    wrote {svg} ({os.path.getsize(svg)/1e6:.2f} MB)")
plt.close(fig3)
print("เสร็จแล้ว")

## เสร็จแล้ว

ไฟล์ที่ได้ใน `out/`: `day06_vintage_plate.png` (แผ่นเต็ม), `day06_vintage_preview.png`,
`day06_roses_plate.png` (rose 24 ดอก), `day06_vintage_lineart.svg` (เส้นหมึกสำหรับ plotter)
และ crop รายละเอียด 3 ภาพ

**สิ่งที่ได้เรียนรู้** — ดูสรุปในโพสต์ [Day 6 · Vintage](https://thammarat-ai.github.io/geo-notes/posts/day06-vintage/)
